# Read data from the bronze layer

In [0]:
dbutils.widgets.text("folderPath","")

filepath = dbutils.widgets.get("folderPath")

In [0]:
df = spark.read.format("parquet").load(filepath)
display(df)

In [0]:
df.printSchema()

# Data transformation

In [0]:
from pyspark.sql.functions import col, split, cast, concat_ws, lpad, try_to_date

In [0]:
df.groupBy(
    "Branch_ID",
    "Dealer_ID",
    "Model_ID",
    "Date_ID"
).count().filter("count > 1").display()

In [0]:
## get model type
df = df.withColumn("ModelType", split(col("Model_ID"),"-")[0])

In [0]:
## rev per unit
df = df.withColumn("RevPerUnit", col("Revenue")/col("Units_Sold"))

In [0]:
## create a proper date column
df = df.withColumn("Date",
              try_to_date(
              concat_ws("-",
              lpad(col("Day").cast("string"),2,"0"),
              lpad(col("Month").cast("string"),2,"0"),
              col("Year").cast("string")),
              "dd-MM-yyyy")
              )

# Isolate bad records

In [0]:
bad_data = df.filter(col("Date").isNull()).drop("Day","Month","Year")
good_data = df.filter(col("Date").isNotNull()).drop("Day","Month","Year")

In [0]:
good_data.groupBy(
    "Branch_ID",
    "Dealer_ID",
    "Model_ID",
    "Date_ID"
).count().filter("count > 1").display()

# Update or Insert records and register as unity catalog table

In [0]:
from delta.tables import DeltaTable

In [0]:
if DeltaTable.isDeltaTable(spark, "abfss://silver@adlscarsales.dfs.core.windows.net/carSales/"):
    deltatable = DeltaTable.forPath(spark, "abfss://silver@adlscarsales.dfs.core.windows.net/carSales/")

    deltatable.alias("trg").merge(good_data.alias("src"),
                                  '''trg.Branch_ID = src.Branch_ID and
                                     trg.Dealer_ID = src.Dealer_ID and
                                     trg.Model_ID = src.Model_ID and
                                     trg.Date_ID = src.Date_ID''')\
                                     .whenMatchedUpdateAll()\
                                     .whenNotMatchedInsertAll()\
                                     .execute()

else:
    good_data.write.format("delta").mode("overwrite").save("abfss://silver@adlscarsales.dfs.core.windows.net/carSales/")

    spark.sql("""
              create table carsalescatalog.silver.carSales
              using delta
              location 'abfss://silver@adlscarsales.dfs.core.windows.net/carSales/'
              """)
                                        

# Append rejected rows

In [0]:
if DeltaTable.isDeltaTable(spark, "abfss://silver@adlscarsales.dfs.core.windows.net/rejectedCarSales/"):
    deltatable = DeltaTable.forPath(spark, "abfss://silver@adlscarsales.dfs.core.windows.net/rejectedCarSales/")

    deltatable.alias("trg").merge(bad_data.alias("src"),
                                  '''trg.Branch_ID = src.Branch_ID and
                                     trg.Dealer_ID = src.Dealer_ID and
                                     trg.Model_ID = src.Model_ID and
                                     trg.Date_ID = src.Date_ID''')\
                                     .whenMatchedUpdateAll()\
                                     .whenNotMatchedInsertAll()\
                                     .execute()

else:
    bad_data.write.format("delta").mode("overwrite").save("abfss://silver@adlscarsales.dfs.core.windows.net/rejectedCarSales/")

    spark.sql("""
              create table carsalescatalog.silver.rejectedCarSales
              using delta
              location 'abfss://silver@adlscarsales.dfs.core.windows.net/rejectedCarSales/'
              """)

In [0]:
spark.read.format("delta").load("abfss://silver@adlscarsales.dfs.core.windows.net/carSales/")\
    .groupBy("Branch_ID", "Dealer_ID", "Model_ID", "Date_ID")\
    .count()\
    .filter("count > 1")\
    .display()  